# Local Feature Ablation Benchmark

Local notebook for two tabular ablations: input modality and feature-size sensitivity. Models: Ridge, LightGBM, Random Forest.


In [1]:
from pathlib import Path
import importlib
import itertools
import os
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "new_notebook":
    ROOT = ROOT.parent

NOTEBOOK_DIR = ROOT / "new_notebook"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import within_dataset_4models
importlib.reload(within_dataset_4models)

from within_dataset_4models import (
    CELL_ID_COL,
    DRUG_ID_COL,
    MODEL_PREPROCESS,
    BenchmarkConfig,
    build_tabular_matrix,
    get_tabular_models,
    make_default_config,
    maybe_sample,
    read_gene_expression,
    read_mordred,
    read_response,
    response_for_split,
    run_random_forest_official_style_model,
    run_tabular_model,
    select_gene_columns,
    top_variance_columns,
)

cfg = make_default_config(ROOT)
cfg


BenchmarkConfig(root=PosixPath('/Users/vietanh/Desktop/ML-Predicting-drug-response'), datasets=['CCLE'], folds=[0], models=['ridge', 'random_forest', 'lightgbm', 'graphdrp', 'simple_linear_nn'], use_lincs_symbol_genes=True, top_ge_features=512, top_mordred_features=512, max_train_rows=None, max_eval_rows=None, random_forest_epochs=100, random_forest_patience=50, graphdrp_epochs=150, graphdrp_batch_size=256, graphdrp_patience=20, graphdrp_learning_rate=0.0001, simple_nn_epochs=300, simple_nn_batch_size=64, simple_nn_val_batch_size=64, simple_nn_patience=50, simple_nn_learning_rate=0.01, simple_nn_dropout=0.01, simple_nn_model='default', random_state=42)

## Config


In [2]:
# Local ablation config.
cfg.datasets = ["CCLE"]
cfg.folds = list(range(3))

# Keep ablation focused and cheaper than the full benchmark.
cfg.models = ["ridge", "lightgbm", "random_forest"]

# Main modality setting: LINCS landmark gene expression + Mordred descriptors.
cfg.use_lincs_symbol_genes = True
cfg.top_ge_features = 512      # ignored when use_lincs_symbol_genes=True
cfg.top_mordred_features = 512

cfg.random_forest_epochs = 100
cfg.random_forest_patience = 50

# Use for quick debug only, then set back to None.
cfg.max_train_rows = None
cfg.max_eval_rows = None

ABLATION_OUT_DIR = cfg.out_dir / "ablations_local"
ABLATION_OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Ablation outputs will be saved to: {ABLATION_OUT_DIR}")
cfg


Ablation outputs will be saved to: /Users/vietanh/Desktop/ML-Predicting-drug-response/new_notebook/results/ablations_local


BenchmarkConfig(root=PosixPath('/Users/vietanh/Desktop/ML-Predicting-drug-response'), datasets=['CCLE'], folds=[0, 1, 2], models=['ridge', 'lightgbm', 'random_forest'], use_lincs_symbol_genes=True, top_ge_features=512, top_mordred_features=512, max_train_rows=None, max_eval_rows=None, random_forest_epochs=100, random_forest_patience=50, graphdrp_epochs=150, graphdrp_batch_size=256, graphdrp_patience=20, graphdrp_learning_rate=0.0001, simple_nn_epochs=300, simple_nn_batch_size=64, simple_nn_val_batch_size=64, simple_nn_patience=50, simple_nn_learning_rate=0.01, simple_nn_dropout=0.01, simple_nn_model='default', random_state=42)

## Ablation Runner


In [3]:
def _build_feature_sets(cfg, train_df, gene_expression, mordred, mode, top_ge=None, top_mordred=None):
    if top_ge is not None:
        ge_cfg = BenchmarkConfig(**{**cfg.__dict__, "use_lincs_symbol_genes": False, "top_ge_features": top_ge})
    else:
        ge_cfg = cfg

    if mode in {"ge_only", "ge_plus_drug"}:
        ge_cols = select_gene_columns(ge_cfg, gene_expression, train_df[CELL_ID_COL].unique())
    else:
        ge_cols = []

    if mode in {"drug_only", "ge_plus_drug"}:
        md_top_k = cfg.top_mordred_features if top_mordred is None else top_mordred
        md_cols = top_variance_columns(mordred, train_df[DRUG_ID_COL].unique(), md_top_k)
    else:
        md_cols = []

    if not ge_cols and not md_cols:
        raise ValueError(f"No features selected for mode={mode!r}")
    return ge_cols, md_cols


def run_tabular_ablation(cfg, ablation_name, settings, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    response = read_response(cfg)
    gene_expression = read_gene_expression(cfg)
    mordred = read_mordred(cfg)
    tabular_models = get_tabular_models(cfg)
    rows_all = []

    for setting in settings:
        setting_name = setting["setting"]
        mode = setting["mode"]
        top_ge = setting.get("top_ge_features")
        top_mordred = setting.get("top_mordred_features")
        print("\n" + "=" * 90)
        print(f"Ablation={ablation_name} | setting={setting_name}")

        for dataset in cfg.datasets:
            for fold in cfg.folds:
                print("-" * 80)
                print(f"Dataset={dataset} | fold={fold}")
                train_df = response_for_split(cfg, response, dataset, fold, "train")
                val_df = response_for_split(cfg, response, dataset, fold, "val")
                test_df = response_for_split(cfg, response, dataset, fold, "test")
                split_sizes = {"train": len(train_df), "val": len(val_df), "test": len(test_df)}

                ge_cols, md_cols = _build_feature_sets(
                    cfg,
                    train_df,
                    gene_expression,
                    mordred,
                    mode=mode,
                    top_ge=top_ge,
                    top_mordred=top_mordred,
                )
                print(f"Features: GE={len(ge_cols):,}, Mordred={len(md_cols):,}, total={len(ge_cols) + len(md_cols):,}")

                X_train, y_train, meta_train = build_tabular_matrix(train_df, gene_expression, mordred, ge_cols, md_cols)
                X_val, y_val, meta_val = build_tabular_matrix(val_df, gene_expression, mordred, ge_cols, md_cols)
                X_test, y_test, meta_test = build_tabular_matrix(test_df, gene_expression, mordred, ge_cols, md_cols)

                X_train, y_train, meta_train = maybe_sample(X_train, y_train, meta_train, cfg.max_train_rows, cfg.random_state)
                X_val, y_val, meta_val = maybe_sample(X_val, y_val, meta_val, cfg.max_eval_rows, cfg.random_state)
                X_test, y_test, meta_test = maybe_sample(X_test, y_test, meta_test, cfg.max_eval_rows, cfg.random_state)

                for model_name in cfg.models:
                    if model_name in {"ridge", "lightgbm"}:
                        if model_name not in tabular_models:
                            rows = [{
                                "analysis": ablation_name,
                                "ablation": ablation_name,
                                "setting": setting_name,
                                "dataset": dataset,
                                "fold": fold,
                                "stage": "test",
                                "model": model_name,
                                "status": "skipped_missing_dependency",
                            }]
                            print(f"  Skipped {model_name}: missing dependency")
                        else:
                            print(f"  Training {model_name}")
                            rows = run_tabular_model(
                                model_name,
                                tabular_models[model_name],
                                dataset,
                                fold,
                                X_train,
                                y_train,
                                X_val,
                                y_val,
                                X_test,
                                y_test,
                            )
                    elif model_name == "random_forest":
                        print("  Training random_forest")
                        rows = run_random_forest_official_style_model(
                            cfg,
                            dataset,
                            fold,
                            X_train,
                            y_train,
                            X_val,
                            y_val,
                            X_test,
                            y_test,
                        )
                    else:
                        raise ValueError(f"This ablation notebook only supports tabular models, got: {model_name}")

                    for row in rows:
                        row["analysis"] = ablation_name
                        row["ablation"] = ablation_name
                        row["setting"] = setting_name
                        row["feature_mode"] = mode
                        row["top_ge_features"] = top_ge
                        row["top_mordred_features"] = top_mordred
                        row["n_ge_features"] = len(ge_cols)
                        row["n_mordred_features"] = len(md_cols)
                        row["preprocess"] = MODEL_PREPROCESS.get(model_name, "custom")
                        row.setdefault("split_train_rows", split_sizes["train"])
                        row.setdefault("split_val_rows", split_sizes["val"])
                        row.setdefault("split_test_rows", split_sizes["test"])
                        if row.get("status") == "ok":
                            print(
                                f"    {row['stage']}: n={row['n']:,} RMSE={row['rmse']:.4f} "
                                f"MAE={row['mae']:.4f} R2={row['r2']:.4f} Pearson={row['pearson']:.4f}"
                            )
                    rows_all.extend(rows)

    results = pd.DataFrame(rows_all)
    results_path = out_dir / f"{ablation_name}_results.csv"
    results.to_csv(results_path, index=False)
    print(f"\nSaved: {results_path}")
    return results


def summarize_ablation_results(results, out_dir, ablation_name):
    ok = results[results.get("status", "ok").eq("ok") & results["stage"].eq("test")].copy()
    if ok.empty:
        return pd.DataFrame()
    summary = (
        ok.groupby(["ablation", "setting", "dataset", "model"], as_index=False)
        .agg(
            folds=("fold", "nunique"),
            n_test_mean=("n", "mean"),
            n_ge_features=("n_ge_features", "mean"),
            n_mordred_features=("n_mordred_features", "mean"),
            r2_mean=("r2", "mean"),
            r2_std=("r2", "std"),
            rmse_mean=("rmse", "mean"),
            rmse_std=("rmse", "std"),
            mae_mean=("mae", "mean"),
            mae_std=("mae", "std"),
            pearson_mean=("pearson", "mean"),
            pearson_std=("pearson", "std"),
        )
        .sort_values(["dataset", "model", "r2_mean"], ascending=[True, True, False])
    )
    path = Path(out_dir) / f"{ablation_name}_summary.csv"
    summary.to_csv(path, index=False)
    print(f"Saved: {path}")
    return summary


## Feature Modality Ablation

Compares gene expression only, drug descriptors only, and combined features.


In [4]:
FEATURE_MODALITY_SETTINGS = [
    {"setting": "gene_expression_only", "mode": "ge_only"},
    {"setting": "drug_descriptors_only", "mode": "drug_only"},
    {"setting": "gene_expression_plus_drug", "mode": "ge_plus_drug"},
]

modality_results = run_tabular_ablation(
    cfg,
    ablation_name="feature_modality_ablation",
    settings=FEATURE_MODALITY_SETTINGS,
    out_dir=ABLATION_OUT_DIR,
)
display(modality_results)

modality_summary = summarize_ablation_results(
    modality_results,
    ABLATION_OUT_DIR,
    "feature_modality_ablation",
)
display(modality_summary)



Ablation=feature_modality_ablation | setting=gene_expression_only
--------------------------------------------------------------------------------
Dataset=CCLE | fold=0
Features: GE=958, Mordred=0, total=958
  Training ridge
    val: n=952 RMSE=0.1563 MAE=0.1212 R2=-0.0245 Pearson=0.1326
    test: n=951 RMSE=0.1664 MAE=0.1261 R2=-0.0165 Pearson=0.1192
  Training lightgbm


/opt/anaconda3/envs/demo/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/demo/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


    val: n=952 RMSE=0.1563 MAE=0.1212 R2=-0.0250 Pearson=0.1324
    test: n=951 RMSE=0.1664 MAE=0.1261 R2=-0.0169 Pearson=0.1190
  Training random_forest
    val: n=952 RMSE=0.1564 MAE=0.1214 R2=-0.0252 Pearson=0.1315
    test: n=951 RMSE=0.1666 MAE=0.1262 R2=-0.0189 Pearson=0.1168
--------------------------------------------------------------------------------
Dataset=CCLE | fold=1
Features: GE=958, Mordred=0, total=958
  Training ridge
    val: n=951 RMSE=0.1665 MAE=0.1260 R2=-0.0184 Pearson=0.1154
    test: n=952 RMSE=0.1592 MAE=0.1215 R2=-0.0160 Pearson=0.1317
  Training lightgbm


/opt/anaconda3/envs/demo/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/demo/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


    val: n=951 RMSE=0.1665 MAE=0.1261 R2=-0.0188 Pearson=0.1152
    test: n=952 RMSE=0.1593 MAE=0.1215 R2=-0.0164 Pearson=0.1316
  Training random_forest
    val: n=951 RMSE=0.1666 MAE=0.1261 R2=-0.0190 Pearson=0.1150
    test: n=952 RMSE=0.1594 MAE=0.1216 R2=-0.0181 Pearson=0.1289
--------------------------------------------------------------------------------
Dataset=CCLE | fold=2
Features: GE=958, Mordred=0, total=958
  Training ridge
    val: n=952 RMSE=0.1592 MAE=0.1214 R2=-0.0153 Pearson=0.1338
    test: n=952 RMSE=0.1658 MAE=0.1285 R2=-0.0236 Pearson=0.1048
  Training lightgbm


...

/opt/anaconda3/envs/demo/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/demo/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


    val: n=952 RMSE=0.0681 MAE=0.0537 R2=0.8141 Pearson=0.9023
    test: n=952 RMSE=0.0715 MAE=0.0547 R2=0.8095 Pearson=0.8999
  Training random_forest
    val: n=952 RMSE=0.0777 MAE=0.0597 R2=0.7581 Pearson=0.8712
    test: n=952 RMSE=0.0820 MAE=0.0623 R2=0.7497 Pearson=0.8660

Saved: /Users/vietanh/Desktop/ML-Predicting-drug-response/new_notebook/results/ablations_local/feature_modality_ablation_results.csv


,analysis,dataset,fold,stage,model,train_seconds,n_train,n_features,status,n,...,setting,feature_mode,top_ge_features,top_mordred_features,n_ge_features,n_mordred_features,preprocess,split_train_rows,split_val_rows,split_test_rows
0,feature_modality_ablation,CCLE,0,val,ridge,0.366444,7616,958,ok,952,...,gene_expression_only,ge_only,None,None,958,0,tabular gene expression + Mordred drug descrip...,7616,952,951
1,feature_modality_ablation,CCLE,0,test,ridge,0.366444,7616,958,ok,951,...,gene_expression_only,ge_only,None,None,958,0,tabular gene expression + Mordred drug descrip...,7616,952,951
2,feature_modality_ablation,CCLE,0,val,lightgbm,16.396022,7616,958,ok,952,...,gene_expression_only,ge_only,None,None,958,0,tabular gene expression + Mordred drug descrip...,7616,952,951
3,feature_modality_ablation,CCLE,0,test,lightgbm,16.396022,7616,958,ok,951,...,gene_expression_only,ge_only,None,None,958,0,tabular gene expression + Mordred drug descrip...,7616,952,951
4,feature_modality_ablation,CCLE,0,val,random_forest,73.529787,7616,958,ok,952,...,gene_expression_only,ge_only,None,None,958,0,official-style tabular gene expression + Mordr...,7616,952,951
5,feature_modality_ablation,CCLE,0,test,random_forest,73.529787,7616,958,ok,951,...,gene_expression_only,ge_only,None,None,958,0,official-style tabular gene expression + Mordr...,7616,952,951
6,feature_modality_ablation,CCLE,1,val,ridge,0.354664,7616,958,ok,951,...,gene_expression_only,ge_only,None,None,958,0,tabular gene expression + Mordred drug descrip...,7616,951,952
7,feature_modality_ablation,CCLE,1,test,ridge,0.354664,7616,958,ok,952,...,gene_expression_only,ge_only,None,None,958,0,tabular gene expression + Mordred drug descrip...,7616,951,952
8,feature_modality_ablation,CCLE,1,val,lightgbm,15.733430,7616,958,ok,951,...,gene_expression_only,ge_only,None,None,958,0,tabular gene expression + Mordred drug descrip...,7616,951,952
9,feature_modality_ablation,CCLE,1,test,lightgbm,15.733430,7616,958,ok,952,...,gene_expression_only,ge_only,None,None,958,0,tabular gene expression + Mordred drug descrip...,7616,951,952


Saved: /Users/vietanh/Desktop/ML-Predicting-drug-response/new_notebook/results/ablations_local/feature_modality_ablation_summary.csv


,ablation,setting,dataset,model,folds,n_test_mean,n_ge_features,n_mordred_features,r2_mean,r2_std,rmse_mean,rmse_std,mae_mean,mae_std,pearson_mean,pearson_std
6,feature_modality_ablation,gene_expression_plus_drug,CCLE,lightgbm,3,951.666667,958.0,512.0,0.809760,0.001732,0.070798,0.001957,0.055032,0.001169,0.900068,0.000773
0,feature_modality_ablation,drug_descriptors_only,CCLE,lightgbm,3,951.666667,0.0,512.0,0.702762,0.019966,0.088472,0.004150,0.069200,0.003569,0.838517,0.011798
3,feature_modality_ablation,gene_expression_only,CCLE,lightgbm,3,951.666667,958.0,0.0,-0.019047,0.004190,0.163849,0.003964,0.125349,0.003542,0.118430,0.013427
7,feature_modality_ablation,gene_expression_plus_drug,CCLE,random_forest,3,951.666667,958.0,512.0,0.746339,0.013689,0.081755,0.003916,0.062804,0.002788,0.864041,0.007975
1,feature_modality_ablation,drug_descriptors_only,CCLE,random_forest,3,951.666667,0.0,512.0,0.702801,0.020089,0.088466,0.004175,0.069186,0.003575,0.838582,0.011785
4,feature_modality_ablation,gene_expression_only,CCLE,random_forest,3,951.666667,958.0,0.0,-0.021929,0.005974,0.164081,0.004043,0.125445,0.003500,0.115036,0.014772
8,feature_modality_ablation,gene_expression_plus_drug,CCLE,ridge,3,951.666667,958.0,512.0,0.745086,0.006962,0.081946,0.002376,0.062692,0.002547,0.863599,0.004129
2,feature_modality_ablation,drug_descriptors_only,CCLE,ridge,3,951.666667,0.0,512.0,0.702761,0.019967,0.088472,0.004150,0.069201,0.003570,0.838518,0.011799
5,feature_modality_ablation,gene_expression_only,CCLE,ridge,3,951.666667,958.0,0.0,-0.018680,0.004264,0.163819,0.003966,0.125330,0.003546,0.118551,0.013457


## Feature Size Ablation

Runs a grid over gene-expression and Mordred feature counts.


In [5]:
# Full grid: 3 x 3 = 9 settings. Reduce these lists for a quick smoke run.
GE_FEATURE_SIZES = [256, 512, 976]
MORDRED_FEATURE_SIZES = [256, 512, 1024]

FEATURE_SIZE_SETTINGS = [
    {
        "setting": f"ge{ge}_mordred{md}",
        "mode": "ge_plus_drug",
        "top_ge_features": ge,
        "top_mordred_features": md,
    }
    for ge, md in itertools.product(GE_FEATURE_SIZES, MORDRED_FEATURE_SIZES)
]

feature_size_results = run_tabular_ablation(
    cfg,
    ablation_name="feature_size_ablation",
    settings=FEATURE_SIZE_SETTINGS,
    out_dir=ABLATION_OUT_DIR,
)
display(feature_size_results)

feature_size_summary = summarize_ablation_results(
    feature_size_results,
    ABLATION_OUT_DIR,
    "feature_size_ablation",
)
display(feature_size_summary)



Ablation=feature_size_ablation | setting=ge256_mordred256
--------------------------------------------------------------------------------
Dataset=CCLE | fold=0
Features: GE=256, Mordred=256, total=512
  Training ridge
    val: n=952 RMSE=0.0842 MAE=0.0642 R2=0.7028 Pearson=0.8389
    test: n=951 RMSE=0.0855 MAE=0.0664 R2=0.7318 Pearson=0.8556
  Training lightgbm


/opt/anaconda3/envs/demo/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/demo/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


    val: n=952 RMSE=0.0711 MAE=0.0556 R2=0.7883 Pearson=0.8881
    test: n=951 RMSE=0.0731 MAE=0.0567 R2=0.8036 Pearson=0.8967
  Training random_forest
    val: n=952 RMSE=0.0797 MAE=0.0620 R2=0.7339 Pearson=0.8568
    test: n=951 RMSE=0.0840 MAE=0.0649 R2=0.7410 Pearson=0.8609
--------------------------------------------------------------------------------
Dataset=CCLE | fold=1
Features: GE=256, Mordred=256, total=512
  Training ridge
    val: n=951 RMSE=0.0856 MAE=0.0668 R2=0.7308 Pearson=0.8550
    test: n=952 RMSE=0.0817 MAE=0.0626 R2=0.7326 Pearson=0.8567
  Training lightgbm


/opt/anaconda3/envs/demo/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/demo/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


    val: n=951 RMSE=0.0722 MAE=0.0564 R2=0.8085 Pearson=0.8994
    test: n=952 RMSE=0.0698 MAE=0.0551 R2=0.8047 Pearson=0.8972
  Training random_forest
    val: n=951 RMSE=0.0833 MAE=0.0645 R2=0.7451 Pearson=0.8634
    test: n=952 RMSE=0.0762 MAE=0.0592 R2=0.7674 Pearson=0.8760
--------------------------------------------------------------------------------
Dataset=CCLE | fold=2
Features: GE=256, Mordred=256, total=512
  Training ridge
    val: n=952 RMSE=0.0814 MAE=0.0624 R2=0.7346 Pearson=0.8579
    test: n=952 RMSE=0.0833 MAE=0.0641 R2=0.7417 Pearson=0.8613
  Training lightgbm


...

/opt/anaconda3/envs/demo/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/demo/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


    val: n=952 RMSE=0.0692 MAE=0.0543 R2=0.7989 Pearson=0.8943
    test: n=951 RMSE=0.0720 MAE=0.0563 R2=0.8093 Pearson=0.9003
  Training random_forest
    val: n=952 RMSE=0.0799 MAE=0.0620 R2=0.7325 Pearson=0.8559
    test: n=951 RMSE=0.0842 MAE=0.0647 R2=0.7394 Pearson=0.8599
--------------------------------------------------------------------------------
Dataset=CCLE | fold=1
Features: GE=976, Mordred=1,024, total=2,000
  Training ridge
    val: n=951 RMSE=0.0846 MAE=0.0658 R2=0.7369 Pearson=0.8585
    test: n=952 RMSE=0.0797 MAE=0.0608 R2=0.7453 Pearson=0.8644
  Training lightgbm


/opt/anaconda3/envs/demo/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/demo/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


    val: n=951 RMSE=0.0708 MAE=0.0556 R2=0.8161 Pearson=0.9041
    test: n=952 RMSE=0.0698 MAE=0.0550 R2=0.8050 Pearson=0.8975
  Training random_forest


KeyboardInterrupt: 

## Ablation Plots


In [ ]:
os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", str(ABLATION_OUT_DIR / ".mplconfig"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-whitegrid")

FIG_DIR = ABLATION_OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


def plot_ablation_errorbars(summary, ablation_name, metric="r2"):
    if summary.empty:
        print(f"No summary for {ablation_name}")
        return
    mean_col = f"{metric}_mean"
    std_col = f"{metric}_std"
    settings = list(summary["setting"].drop_duplicates())
    models = list(summary["model"].drop_duplicates())
    datasets = list(summary["dataset"].drop_duplicates())

    for dataset in datasets:
        df = summary[summary["dataset"].eq(dataset)].copy()
        x = np.arange(len(settings))
        width = min(0.22, 0.75 / max(len(models), 1))
        fig, ax = plt.subplots(figsize=(max(8, 1.2 * len(settings)), 4.8), constrained_layout=True)
        for i, model in enumerate(models):
            sub = df[df["model"].eq(model)].set_index("setting").reindex(settings)
            offset = (i - (len(models) - 1) / 2) * width
            ax.bar(x + offset, sub[mean_col], width=width, yerr=sub[std_col], capsize=3, label=model, alpha=0.9)
        ax.set_xticks(x)
        ax.set_xticklabels(settings, rotation=35, ha="right")
        ylabel = {"r2": "Test R2", "rmse": "Test RMSE", "pearson": "Test Pearson r"}.get(metric, metric)
        ax.set_ylabel(f"{ylabel} mean +/- std")
        ax.set_title(f"{ablation_name}: {ylabel} ({dataset})")
        ax.legend(title="Model")
        path = FIG_DIR / f"{ablation_name}_{metric}_{dataset}.png"
        fig.savefig(path, bbox_inches="tight", dpi=300)
        print(f"Saved: {path}")
        plt.show()


def plot_feature_size_heatmap(summary, model="lightgbm", dataset=None, metric="r2_mean"):
    if summary.empty:
        return
    df = summary[summary["model"].eq(model)].copy()
    if df.empty:
        return
    if dataset is None:
        dataset = df["dataset"].iloc[0]
    df = df[df["dataset"].eq(dataset)].copy()
    if df.empty:
        print(f"No rows for model={model}, dataset={dataset}")
        return
    matrix = df.pivot(index="n_ge_features", columns="n_mordred_features", values=metric).sort_index().sort_index(axis=1)
    fig, ax = plt.subplots(figsize=(6.2, 4.8), constrained_layout=True)
    im = ax.imshow(matrix.to_numpy(), cmap="YlGnBu", aspect="auto")
    ax.set_xticks(np.arange(matrix.shape[1]))
    ax.set_xticklabels([int(c) for c in matrix.columns])
    ax.set_yticks(np.arange(matrix.shape[0]))
    ax.set_yticklabels([int(i) for i in matrix.index])
    ax.set_xlabel("Mordred descriptor count")
    ax.set_ylabel("Gene-expression feature count")
    ax.set_title(f"Feature-size ablation: {model} {metric} ({dataset})")
    threshold = np.nanmean(matrix.to_numpy())
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix.iloc[i, j]
            ax.text(j, i, f"{value:.3f}", ha="center", va="center", color="white" if value >= threshold else "#1f2933")
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label(metric)
    path = FIG_DIR / f"feature_size_heatmap_{model}_{dataset}_{metric}.png"
    fig.savefig(path, bbox_inches="tight", dpi=300)
    print(f"Saved: {path}")
    plt.show()

plot_ablation_errorbars(modality_summary, "feature_modality_ablation", metric="r2")
plot_ablation_errorbars(modality_summary, "feature_modality_ablation", metric="rmse")
plot_ablation_errorbars(feature_size_summary, "feature_size_ablation", metric="r2")

for model in cfg.models:
    plot_feature_size_heatmap(feature_size_summary, model=model, metric="r2_mean")
